In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from sklearn.metrics import confusion_matrix, cohen_kappa_score
import os


In [ ]:
LLM_IDENTIFIERS = ["Claude", "Gemini", "GPT-4.1", "GPT-o3"]
CATEGORY_HIERARCHY_REACTIONS = {
    "Reaction feasible, all good": 5,  # Most positive
    "Reaction feasible, unexpected disconnection": 5,  
    "Protecting group strategy is wrong / non-optimal": 3,  
    "Non-optimal reagent": 4,
    "Unnecessary step":4,
    "Selectivity (regio-, stereo-, chemo-) issues": 2,
    "Problems with reaction type and functional group compatibility": 2,  
    "Unlikely disconnection": 1  # Most negative
}

# Sentiment categories
SENTIMENT_MAPPING_REACTIONS = {
    "Reaction feasible, all good": "Positive",
    "Reaction feasible, unexpected disconnection": "Positive", 
    "Non-optimal reagent": "Negative",
    "Unnecessary step":"Negative",
    "Protecting group strategy is wrong / non-optimal": "Negative",
    "Selectivity (regio-, stereo-, chemo-) issues": "Negative",
    "Problems with reaction type and functional group compatibility": "Negative",
    "Unlikely disconnection": "Negative"
}
aliases = {
    "Reaction feasible, all good": "Reaction feasible",
    "Reaction feasible, unexpected disconnection": "Unexpected disconnection", 
    "Non-optimal reagent": "Non-optimal reagent",
    "Unnecessary step":"Unnecessary step",
    "Protecting group strategy is wrong / non-optimal": "Protecting group issues",
    "Selectivity (regio-, stereo-, chemo-) issues": "Selectivity issues",
    "Problems with reaction type and functional group compatibility": "Functional group issues",
    "Unlikely disconnection": "Unlikely disconnection"
}
# Define category hierarchy (from most positive to most negative)
CATEGORY_HIERARCHY_ROUTES = {
    "Route feasible": 5,  # Most positive
    "Route feasible with few modifications": 4,  
    "Route feasible with significant modifications": 2,  
    "Route unfeasible": 1,
    "Route was not solved to building blocks":1,
}

# Define sentiment categories
SENTIMENT_MAPPING_ROUTES = {
    "Route feasible": "Positive",
    "Route feasible with few modifications": "Positive", 
    "Route feasible with significant modifications": "Negative",
    "Route unfeasible":"Negative",
    "Route was not solved to building blocks": "Negative",
}


In [ ]:
LLM_IDENTIFIERS = ["Claude", "Gemini", "GPT-4.1", "GPT-o3"]

class CategoryMappings:
    """Container for category mappings and hierarchy for different analysis types"""
    
    REACTIONS = {
        'hierarchy': {
            "Reaction feasible, all good": 5,
            "Reaction feasible, unexpected disconnection": 5,
            "Non-optimal reagent": 4,
            "Unnecessary step": 4,
            "Protecting group strategy is wrong / non-optimal": 3,
            "Selectivity (regio-, stereo-, chemo-) issues": 2,
            "Problems with reaction type and functional group compatibility": 2,
            "Unlikely disconnection": 1
        },
        'sentiment': {
            "Reaction feasible, all good": "Positive",
            "Reaction feasible, unexpected disconnection": "Positive",
            "Non-optimal reagent": "Negative",
            "Unnecessary step": "Negative",
            "Protecting group strategy is wrong / non-optimal": "Negative",
            "Selectivity (regio-, stereo-, chemo-) issues": "Negative",
            "Problems with reaction type and functional group compatibility": "Negative",
            "Unlikely disconnection": "Negative"
        },
        'aliases': {
            "Reaction feasible, all good": "Reaction feasible",
            "Reaction feasible, unexpected disconnection": "Unexpected disconnection",
            "Non-optimal reagent": "Non-optimal reagent",
            "Unnecessary step": "Unnecessary step",
            "Protecting group strategy is wrong / non-optimal": "Protecting group issues",
            "Selectivity (regio-, stereo-, chemo-) issues": "Selectivity issues",
            "Problems with reaction type and functional group compatibility": "Functional group issues",
            "Unlikely disconnection": "Unlikely disconnection"
        }
    }
    
    ROUTES = {
        'hierarchy': {
            "Route feasible as it is": 5,
            "Route feasible with few modifications": 4,
            "Route feasible with significant modifications": 2,
            "Route unfeasible": 1,
            "Route was not solved to building blocks": 1,
        },
        'sentiment': {
            "Route feasible as it is": "Positive",
            "Route feasible with few modifications": "Positive",
            "Route feasible with significant modifications": "Negative",
            "Route unfeasible": "Negative",
            "Route was not solved to building blocks": "Negative",
        },
        'aliases': {
            "Route feasible as it is": "Feasible as it is",
            "Route feasible with few modifications": "Few modifications",
            "Route feasible with significant modifications": "Significant modifications",
            "Route unfeasible": "Unfeasible",
            "Route was not solved to building blocks": "Not solved",
        }
    }
    
    @classmethod
    def get_mappings(cls, hash_column):
        """Get the appropriate mappings based on hash_column"""
        if hash_column == 'reaction_hash':
            return cls.REACTIONS
        else:
            return cls.ROUTES


class ConsensusAnalyzer:
    """Handles consensus analysis with pessimistic tie-breaking"""
    
    def __init__(self, hash_column):
        self.mappings = CategoryMappings.get_mappings(hash_column)
        self.hierarchy = self.mappings['hierarchy']
        self.sentiment_mapping = self.mappings['sentiment']
    
    def _pick_pessimistic_category(self, categories):
        """Pick the most pessimistic category from a list"""
        if not categories:
            return None
        # Lower hierarchy value = worse (more pessimistic)
        # For categories missing from hierarchy, assign -inf so they sort first (most pessimistic)
        return sorted(categories, key=lambda c: self.hierarchy.get(c, -np.inf))[0]
    
    def get_majority_category_with_pessimism(self, categories):
        """
        Apply 3-step rule for consensus with pessimistic tie-breaking:
        1) If strict majority exists, choose it
        2) Else, compute sentiment majority; choose categories within that sentiment
           with max votes; if multiple tie, choose 'worse' per hierarchy
        3) If sentiment is tied, apply pessimistic tie-breaking across all tied categories
        
        Returns: (chosen_category, chosen_sentiment, votes_by_category, votes_by_sentiment)
        """
        if not categories:
            return "No feedback", "Unknown", Counter(), Counter()

        cat_counts = Counter(categories)
        total = sum(cat_counts.values())
        
        # Step 1: Check for strict majority
        top_cat, top_count = cat_counts.most_common(1)[0]
        if top_count > total / 2.0:
            sent_counts = Counter(self.sentiment_mapping.get(c, "Unknown") for c in categories)
            return top_cat, self.sentiment_mapping.get(top_cat, "Unknown"), cat_counts, sent_counts

        # Build sentiment counts
        sentiments = [self.sentiment_mapping.get(c, "Unknown") for c in categories]
        sent_counts = Counter(sentiments)

        # Step 2: Check for single sentiment (no sentiment competition)
        if len(sent_counts) == 1:
            max_count = max(cat_counts.values())
            tied = [c for c, v in cat_counts.items() if v == max_count]
            choice = self._pick_pessimistic_category(tied)
            return choice, self.sentiment_mapping.get(choice, "Unknown"), cat_counts, sent_counts

        # Check for sentiment majority (clear winner)
        sent_sorted = sent_counts.most_common()
        sent_top, sent_top_count = sent_sorted[0]
        sent_second_count = sent_sorted[1][1] if len(sent_sorted) > 1 else 0

        if sent_top_count != sent_second_count:
            # Sentiment majority exists - pick best category within winning sentiment
            within_sentiment = {c: v for c, v in cat_counts.items() 
                              if self.sentiment_mapping.get(c, "Unknown") == sent_top}
            max_within = max(within_sentiment.values())
            tied_within = [c for c, v in within_sentiment.items() if v == max_within]
            choice = self._pick_pessimistic_category(tied_within)
            return choice, sent_top, cat_counts, sent_counts

        # Step 3: Sentiment tie - pessimistic across all categories tied at max count
        max_count = max(cat_counts.values())
        tied_all = [c for c, v in cat_counts.items() if v == max_count]
        choice = self._pick_pessimistic_category(tied_all)
        return choice, self.sentiment_mapping.get(choice, "Unknown"), cat_counts, sent_counts


class GroupConsensusAnalyzer:
    """Handles group consensus analysis for humans vs LLMs"""
    
    def __init__(self, hash_column):
        self.consensus_analyzer = ConsensusAnalyzer(hash_column)
        self.mappings = CategoryMappings.get_mappings(hash_column)
    
    def get_group_consensus(self, group_data, expert_column, category_column, 
                          text_column=None, confidence_column=None):
        """
        Get consensus information for a group (human or single LLM) on a single reaction
        """
        experts = group_data[expert_column].unique()
        num_experts = len(experts)

        # Detect LLM group (single LLM name) vs humans
        is_llm = (len(experts) == 1) and (experts[0] in LLM_IDENTIFIERS)

        if is_llm:
            # LLM: votes are all rows' categories (re-runs)
            votes = group_data[category_column].dropna().tolist()
            num_voters = len(votes)
            
            if num_voters == 0:
                return self._empty_consensus_result()
            
            dominant_category, dominant_sentiment, cat_counts, _ = \
                self.consensus_analyzer.get_majority_category_with_pessimism(votes)
            
            agreement_pct = (cat_counts.get(dominant_category, 0) / num_voters * 100.0)
            category_breakdown = {
                c: {'count': cnt, 'percentage': (cnt / num_voters * 100.0)} 
                for c, cnt in cat_counts.items()
            }
            
            all_text_feedback = []
            if text_column and text_column in group_data.columns:
                all_text_feedback = group_data[text_column].dropna().tolist()
            
            return {
                'dominant_category': dominant_category,
                'dominant_sentiment': dominant_sentiment,
                'agreement_pct': agreement_pct,
                'num_experts': num_voters,  # number of votes (re-runs)
                'category_breakdown': category_breakdown,
                'text_feedback': all_text_feedback,
                'all_categories': dict(cat_counts)
            }
        
        else:
            # Humans: PRESENCE VOTING (each expert contributes at most one vote per category)
            category_counts = defaultdict(int)
            all_text_feedback = []
            
            for expert in experts:
                expert_rows = group_data[group_data[expert_column] == expert]
                
                # Each category counted once per expert (presence vote)
                expert_categories = set(expert_rows[category_column].dropna().tolist())
                for cat in expert_categories:
                    category_counts[cat] += 1
                
                # Optional text aggregation
                if text_column and text_column in expert_rows.columns:
                    expert_texts = expert_rows[text_column].dropna().tolist()
                    for txt in expert_texts:
                        if isinstance(txt, str) and txt.strip():
                            all_text_feedback.append({'expert': expert, 'text': txt.strip()})
            
            if not category_counts:
                return self._empty_consensus_result()
            
            # Build expanded vote list from presence counts and resolve with pessimistic tie-breaking
            expanded_votes = [cat for cat, count in category_counts.items() for _ in range(count)]
            dominant_category, dominant_sentiment, _, _ = \
                self.consensus_analyzer.get_majority_category_with_pessimism(expanded_votes)
            
            # Agreement percentage over number of experts
            agreement_pct = (category_counts.get(dominant_category, 0) / num_experts * 100.0)
            
            # Breakdown (per expert basis)
            category_breakdown = {
                c: {'count': count, 'percentage': (count / num_experts * 100.0)}
                for c, count in category_counts.items()
            }
            
            return {
                'dominant_category': dominant_category,
                'dominant_sentiment': dominant_sentiment,
                'agreement_pct': agreement_pct,
                'num_experts': num_experts,  # unique humans contributing
                'category_breakdown': category_breakdown,
                'text_feedback': all_text_feedback,
                'all_categories': dict(category_counts)
            }
    
    def _empty_consensus_result(self):
        """Return empty consensus result structure"""
        return {
            'dominant_category': "No feedback",
            'dominant_sentiment': "Unknown",
            'agreement_pct': 0.0,
            'num_experts': 0,
            'category_breakdown': {},
            'text_feedback': [],
            'all_categories': {}
        }


class HumanLLMComparator:
    """Handles comparison between human and LLM evaluations"""
    
    def __init__(self, hash_column):
        self.mappings = CategoryMappings.get_mappings(hash_column)
        self.hierarchy = self.mappings['hierarchy']
        self.sentiment_mapping = self.mappings['sentiment']
    
    def compare_group_evaluations(self, human_analysis, llm_analysis):
        """Compare human vs LLM consensus with sentiment and score analysis"""
        human_category = human_analysis['dominant_category']
        llm_category = llm_analysis['dominant_category']

        human_sentiment = self.sentiment_mapping.get(human_category, 'Unknown')
        llm_sentiment = self.sentiment_mapping.get(llm_category, 'Unknown')

        human_score = self.hierarchy.get(human_category, 0)
        llm_score = self.hierarchy.get(llm_category, 0)

        sentiment_agreement = human_sentiment == llm_sentiment
        category_agreement = human_category == llm_category
        score_difference = abs(human_score - llm_score)

        if category_agreement:
            disagreement_severity = "No disagreement"
        elif sentiment_agreement:
            disagreement_severity = "Mild disagreement (same sentiment)"
        elif score_difference <= 1:
            disagreement_severity = "Moderate disagreement"
        else:
            disagreement_severity = "Strong disagreement"

        return {
            'sentiment_agreement': sentiment_agreement,
            'category_agreement': category_agreement,
            'score_difference': score_difference,
            'disagreement_severity': disagreement_severity,
            'human_sentiment': human_sentiment,
            'llm_sentiment': llm_sentiment,
            'human_score': human_score,
            'llm_score': llm_score
        }


In [ ]:
def generate_confusion_matrices(results_df):
    """Generate confusion matrices for different levels of comparison"""
    # Category-level confusion matrix
    human_categories = results_df['human_dominant_category'].tolist()
    llm_categories = results_df['llm_dominant_category'].tolist()
    
    # Get all unique categories
    all_categories = sorted(list(set(human_categories + llm_categories)))
    
    # Create confusion matrix
    cm_categories = confusion_matrix(human_categories, llm_categories, labels=all_categories)
    
    # Sentiment-level confusion matrix
    human_sentiments = results_df['human_sentiment'].tolist()
    llm_sentiments = results_df['llm_sentiment'].tolist()
    
    sentiment_labels = ['Positive', 'Negative']
    cm_sentiments = confusion_matrix(human_sentiments, llm_sentiments, labels=sentiment_labels)
    
    # Score-level data
    human_scores = results_df['human_score'].tolist()
    llm_scores = results_df['llm_score'].tolist()
    
    return {
        'category_matrix': {
            'matrix': cm_categories,
            'labels': all_categories
        },
        'sentiment_matrix': {
            'matrix': cm_sentiments,
            'labels': sentiment_labels
        },
        'human_categories': human_categories,
        'llm_categories': llm_categories,
        'human_scores': human_scores,
        'llm_scores': llm_scores
    }


def calculate_agreement_statistics(results_df):
    """Calculate various agreement statistics"""
    total_reactions = len(results_df)
    
    if total_reactions == 0:
        return {
            'total_reactions': 0,
            'category_agreement_count': 0,
            'category_agreement_percentage': 0.0,
            'sentiment_agreement_count': 0,
            'sentiment_agreement_percentage': 0.0,
            'disagreement_distribution': {},
            'disagreement_distribution_percentage': {},
            'cohens_kappa': 0.0,
            'mean_score_difference': 0.0,
            'std_score_difference': 0.0,
            'max_score_difference': 0
        }
    
    # Category agreement
    category_agreement = results_df['category_agreement'].sum()
    category_agreement_pct = (category_agreement / total_reactions) * 100
    
    # Sentiment agreement
    sentiment_agreement = results_df['sentiment_agreement'].sum()
    sentiment_agreement_pct = (sentiment_agreement / total_reactions) * 100
    
    # Disagreement severity distribution
    disagreement_dist = results_df['disagreement_severity'].value_counts()
    disagreement_dist_pct = (disagreement_dist / total_reactions * 100).round(1)
    
    # Cohen's Kappa for categories
    human_categories = results_df['human_dominant_category'].tolist()
    llm_categories = results_df['llm_dominant_category'].tolist()
    
    # Convert categories to numeric for kappa calculation
    all_categories = sorted(set(human_categories + llm_categories))
    if len(all_categories) > 1:
        category_to_num = {cat: i for i, cat in enumerate(all_categories)}
        human_numeric = [category_to_num[cat] for cat in human_categories]
        llm_numeric = [category_to_num[cat] for cat in llm_categories]
        kappa_score = cohen_kappa_score(human_numeric, llm_numeric)
    else:
        kappa_score = 1.0  # Perfect agreement if only one category
    
    # Score difference statistics
    score_differences = results_df['score_difference']
    
    return {
        'total_reactions': total_reactions,
        'category_agreement_count': int(category_agreement),
        'category_agreement_percentage': round(category_agreement_pct, 1),
        'sentiment_agreement_count': int(sentiment_agreement),
        'sentiment_agreement_percentage': round(sentiment_agreement_pct, 1),
        'disagreement_distribution': disagreement_dist.to_dict(),
        'disagreement_distribution_percentage': disagreement_dist_pct.to_dict(),
        'cohens_kappa': round(kappa_score, 3),
        'mean_score_difference': round(score_differences.mean(), 2),
        'std_score_difference': round(score_differences.std(), 2),
        'max_score_difference': int(score_differences.max()) if len(score_differences) > 0 else 0
    }


def analyze_disagreements(results_df):
    """Analyze patterns in disagreements"""
    # Filter for disagreements
    disagreements = results_df[~results_df['category_agreement']].copy()
    
    if disagreements.empty:
        return {'message': 'No disagreements found'}
    
    # Most common disagreement patterns
    disagreement_patterns = []
    for _, row in disagreements.iterrows():
        pattern = f"Human: {row['human_dominant_category']} → LLM: {row['llm_dominant_category']}"
        disagreement_patterns.append(pattern)
    
    pattern_counts = Counter(disagreement_patterns)
    
    # Reactions with strongest disagreements
    strongest_disagreements = disagreements.nlargest(10, 'score_difference')
    
    # Sentiment flip analysis
    sentiment_flips = disagreements[~disagreements['sentiment_agreement']]
    
    return {
        'total_disagreements': len(disagreements),
        'disagreement_percentage': round((len(disagreements) / len(results_df)) * 100, 1),
        'most_common_patterns': dict(pattern_counts.most_common(10)),
        'strongest_disagreements': strongest_disagreements[[
            'reaction_hash', 'human_dominant_category', 'llm_dominant_category',  
            'score_difference', 'disagreement_severity'
        ]].to_dict('records'),
        'sentiment_flips': len(sentiment_flips),
        'sentiment_flip_percentage': round((len(sentiment_flips) / len(disagreements)) * 100, 1) if len(disagreements) > 0 else 0
    }


def compare_category_distributions(human_feedback, llm_feedback, category_column, expert_column):
    """Compare overall category distributions between humans and LLMs"""
    # Get category distributions
    human_categories = human_feedback[category_column].value_counts()
    llm_categories = llm_feedback[category_column].value_counts()
    
    # Normalize to percentages
    human_pct = (human_categories / human_categories.sum() * 100).round(1) if len(human_categories) > 0 else pd.Series()
    llm_pct = (llm_categories / llm_categories.sum() * 100).round(1) if len(llm_categories) > 0 else pd.Series()
    
    # Combine into comparison DataFrame
    all_categories = sorted(set(human_categories.index.tolist() + llm_categories.index.tolist()))
    
    comparison_data = []
    for category in all_categories:
        comparison_data.append({
            'category': category,
            'human_count': human_categories.get(category, 0),
            'human_percentage': human_pct.get(category, 0),
            'llm_count': llm_categories.get(category, 0),
            'llm_percentage': llm_pct.get(category, 0),
            'difference': human_pct.get(category, 0) - llm_pct.get(category, 0)
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('difference', key=abs, ascending=False)
    
    return comparison_df


def analyze_human_vs_llm_agreement(feedback_df, hash_column='reaction_hash',
                                 category_column='local_feedback',
                                 expert_column='source_file',
                                 text_column='local_feedback_text',
                                 confidence_column='confidence'):
    """
    Comprehensive analysis comparing human majority vs EACH LLM majority,
    using pessimistic tie-breaking for consensus selection on both sides.
    """
    # Initialize analyzers based on hash_column
    group_analyzer = GroupConsensusAnalyzer(hash_column)
    comparator = HumanLLMComparator(hash_column)
    
    # Separate human and LLM feedback
    feedback_df = feedback_df.dropna(subset=[hash_column])
    
    human_feedback = feedback_df[~feedback_df[expert_column].isin(LLM_IDENTIFIERS)].copy()
    llm_feedback = feedback_df[feedback_df[expert_column].isin(LLM_IDENTIFIERS)].copy()

    print(f"Human experts: {len(human_feedback[expert_column].unique())}")
    print(f"LLM experts: {len(llm_feedback[expert_column].unique())}")
    print(f"Total human feedback entries: {len(human_feedback)}")
    print(f"Total LLM feedback entries: {len(llm_feedback)}")

    # Reactions overlap
    human_reactions = set(human_feedback[hash_column].unique())
    llm_reactions = set(llm_feedback[hash_column].unique())
    common_reactions = human_reactions.intersection(llm_reactions)

    print(f"Reactions evaluated by humans only: {len(human_reactions - llm_reactions)}")
    print(f"Reactions evaluated by LLMs only: {len(llm_reactions - human_reactions)}")
    print(f"Reactions evaluated by both: {len(common_reactions)}")

    if not common_reactions:
        print("No reactions found that were evaluated by both humans and LLMs.")
        return {}

    # Precompute human consensus per reaction once
    human_consensus_by_rxn = {}
    for rxn in common_reactions:
        hdata = human_feedback[human_feedback[hash_column] == rxn]
        human_consensus_by_rxn[rxn] = group_analyzer.get_group_consensus(
            hdata, expert_column, category_column, text_column, confidence_column
        )

    # Analyze for each LLM separately
    comparison_results = []
    for llm_name in LLM_IDENTIFIERS:
        llm_df = llm_feedback[llm_feedback[expert_column] == llm_name]
        llm_rxns = set(llm_df[hash_column].unique())
        rxns_to_compare = sorted(common_reactions.intersection(llm_rxns))

        for reaction_hash in rxns_to_compare:
            # Human consensus (precomputed)
            human_analysis = human_consensus_by_rxn[reaction_hash]

            # LLM consensus (re-runs for this LLM on this reaction)
            llm_data = llm_df[llm_df[hash_column] == reaction_hash]
            llm_analysis = group_analyzer.get_group_consensus(
                llm_data, expert_column, category_column, text_column, confidence_column
            )

            # Compare majority human vs majority LLM
            comparison = comparator.compare_group_evaluations(human_analysis, llm_analysis)

            result = {
                'llm_name': llm_name,
                'reaction_hash': reaction_hash,
                'human_dominant_category': human_analysis['dominant_category'],
                'human_agreement_pct': human_analysis['agreement_pct'],
                'human_sentiment': comparison['human_sentiment'],
                'human_score': comparison['human_score'],
                'human_experts': human_analysis['num_experts'],
                'llm_dominant_category': llm_analysis['dominant_category'],
                'llm_agreement_pct': llm_analysis['agreement_pct'],
                'llm_sentiment': comparison['llm_sentiment'],
                'llm_score': comparison['llm_score'],
                'llm_num_runs': llm_analysis['num_experts'],
                'sentiment_agreement': comparison['sentiment_agreement'],
                'category_agreement': comparison['category_agreement'],
                'score_difference': comparison['score_difference'],
                'disagreement_severity': comparison['disagreement_severity'],
                'human_category_breakdown': human_analysis['category_breakdown'],
                'llm_category_breakdown': llm_analysis['category_breakdown']
            }

            comparison_results.append(result)

    # Create DataFrame
    results_df = pd.DataFrame(comparison_results)

    # Build per-LLM sentiment confusion counts
    matrix_rows = []
    for llm_name in LLM_IDENTIFIERS:
        sub = results_df[results_df['llm_name'] == llm_name]
        if len(sub) > 0:
            TP = int(((sub['human_sentiment'] == 'Positive') & (sub['llm_sentiment'] == 'Positive')).sum())
            TN = int(((sub['human_sentiment'] == 'Negative') & (sub['llm_sentiment'] == 'Negative')).sum())
            FP = int(((sub['human_sentiment'] == 'Negative') & (sub['llm_sentiment'] == 'Positive')).sum())
            FN = int(((sub['human_sentiment'] == 'Positive') & (sub['llm_sentiment'] == 'Negative')).sum())
        else:
            TP = TN = FP = FN = 0
        matrix_rows.append({'llm_name': llm_name, 'TN': TN, 'FN': FN, 'FP': FP, 'TP': TP})
    matrix_df = pd.DataFrame(matrix_rows)

    # Calculate agreement statistics
    agreement_statistics = calculate_agreement_statistics(results_df)

    # Category distribution comparison
    category_comparison = compare_category_distributions(
        human_feedback, llm_feedback, category_column, expert_column
    )

    # Confusion matrices
    confusion_matrix_bundle = generate_confusion_matrices(results_df)

    analysis = {
        'results_df': results_df,
        'matrix_df': matrix_df,
        'confusion_matrix': confusion_matrix_bundle,
        'agreement_statistics': agreement_statistics,
        'disagreement_analysis': analyze_disagreements(results_df),
        'category_comparison': category_comparison
    }
    
    return analysis


def run_human_vs_llm_analysis(feedback_csv, hash_column='reaction_hash', 
                             output_dir='human_vs_llm_analysis',
                             category_column='local_feedback',
                             expert_column='source_file',
                             text_column='local_feedback_text',
                             confidence_column='confidence'):
    """
    Run complete human vs LLM analysis with specified hash_column
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Load data
    feedback_df = pd.read_csv(feedback_csv)
    
    print("Starting Human vs LLM Agreement Analysis...")
    print("=" * 50)
    print(f"Analysis type: {'Reactions' if hash_column == 'reaction_hash' else 'Routes'}")
    
    # Run analysis
    analysis_results = analyze_human_vs_llm_agreement(
        feedback_df, 
        hash_column=hash_column,
        category_column=category_column,
        expert_column=expert_column,
        text_column=text_column,
        confidence_column=confidence_column
    )
    
    if not analysis_results:
        print("Analysis failed. Please check your data.")
        return
    
    # Save results
    results_df = analysis_results['results_df']
    results_df.to_csv(os.path.join(output_dir, 'human_vs_llm_comparison.csv'), index=False)
    
    # Save disagreement details
    disagreement_analysis = analysis_results['disagreement_analysis']
    if 'strongest_disagreements' in disagreement_analysis:
        disagreement_df = pd.DataFrame(disagreement_analysis['strongest_disagreements'])
        disagreement_df.to_csv(os.path.join(output_dir, 'strongest_disagreements.csv'), index=False)
    
    # Save category comparison
    category_comparison = analysis_results['category_comparison']
    category_comparison.to_csv(os.path.join(output_dir, 'category_distribution_comparison.csv'), index=False)
    
    print(f"\nAnalysis complete! Results saved to {output_dir}/")
    
    return analysis_results

In [ ]:
results = run_human_vs_llm_analysis(
    'expert_feedback_combined_llms.csv', 
    hash_column='reaction_hash'
)

In [ ]:
results = run_human_vs_llm_analysis(
    'expert_feedback_combined_llms.csv', 
    hash_column='root_molecule_hash',
    category_column='general_feedback',
    expert_column='source_file',
    text_column='general_feedback_text',
    output_dir='human_vs_llm_analysis_routes',
)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

def _safe_mkdir(path):
    os.makedirs(path, exist_ok=True)
    return path

def save_per_llm_outputs(analysis_results, output_dir, hash_column='reaction_hash',
                        llm_order=("GPT-4.1", "Claude", "GPT-o3", "Gemini")):
    """
    Save all outputs per LLM:
      - per-LLM results_df subset
      - per-LLM stats row
      - per-LLM disagreement subset
      - per-LLM confusion matrices (category & sentiment)
      - per-LLM single-row TN/FN/FP/TP heatmap
    Also saves a combined TN/FN/FP/TP matrix across LLMs (multi-row) like the example image.
    """
    _safe_mkdir(output_dir)
    
    # Get mappings for aliases based on hash_column
    mappings = CategoryMappings.get_mappings(hash_column)
    aliases = mappings.get('aliases', {})
    
    results_df = analysis_results['results_df']
    matrix_df = analysis_results['matrix_df']      # TN/FN/FP/TP per LLM
    
    # Generate stats_df from results if not present
    stats_df = _generate_per_llm_stats(results_df) if 'stats_df' not in analysis_results else analysis_results['stats_df']

    # Combined TN/FN/FP/TP heatmap across all LLMs (multi-row)
    # Reorder according to llm_order if present in matrix_df
    matrix_df = matrix_df.set_index('llm_name')
    available_llms = [llm for llm in llm_order if llm in matrix_df.index]
    matrix_df = matrix_df.loc[available_llms].reset_index()
    llms = matrix_df['llm_name'].tolist()
    mat = matrix_df[['TN','FN','FP','TP']].values.astype(int)

    # Create combined agreement matrix
    plt.figure(figsize=(10, 8))
    im = plt.imshow(mat, cmap='RdYlGn', aspect='auto')
    analysis_type = 'Reaction' if hash_column == 'reaction_hash' else 'Route'
    plt.title(f'Confusion Matrix\n({analysis_type} Counts)', fontsize=18, fontweight='bold')
    plt.xticks(range(4), ['TN', 'FN', 'FP', 'TP'], fontsize=16)
    plt.yticks(range(len(llms)), llms, fontsize=12)
    cbar = plt.colorbar(im)
    cbar.set_label('Count', rotation=90)
    max_val = mat.max() if mat.size > 0 else 1
    thresh = max_val / 2.0
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = int(mat[i, j])
            plt.text(j, i, f'{val}', ha='center', va='center',
                    color='white' if val > thresh else 'black',
                    fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'agreement_matrix_all_llms_{analysis_type}.png'), dpi=300, bbox_inches='tight')
    plt.close()

    # Per-LLM outputs
    for llm_name in llms:
        llm_dir = _safe_mkdir(os.path.join(output_dir, f'llm_{llm_name}'))

        # Per-LLM comparisons CSV
        sub = results_df[results_df['llm_name'] == llm_name].copy()
        sub.to_csv(os.path.join(llm_dir, f'human_vs_{llm_name}_comparisons_{analysis_type}.csv'), index=False)

        # Per-LLM TN/FN/FP/TP row and single-row heatmap
        row = matrix_df[matrix_df['llm_name'] == llm_name][['TN','FN','FP','TP']].values.astype(int).flatten()
        plt.figure(figsize=(7, 2.8))
        im = plt.imshow(row.reshape(1, -1), cmap='RdYlGn', aspect='auto')
        plt.title(f'Confusion Matrix (Counts) – {llm_name}', fontsize=14, fontweight='bold')
        plt.xticks(range(4), ['TN','FN','FP','TP'], fontsize=12)
        plt.yticks([0], [llm_name], fontsize=12)
        cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
        cbar.set_label('Count', rotation=90)
        max_val = row.max() if row.size > 0 else 1
        thresh = max_val / 2.0
        for j in range(4):
            val = int(row[j])
            plt.text(j, 0, f'{val}', ha='center', va='center',
                    color='white' if val > thresh else 'black',
                    fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(llm_dir, f'agreement_matrix_{llm_name}_{analysis_type}.png'), dpi=300, bbox_inches='tight')
        plt.close()

        # Per-LLM confusion matrices for categories and sentiment
        # Build confusion matrices from per-LLM subset
        hc = sub['human_dominant_category'].tolist()
        lc = sub['llm_dominant_category'].tolist()

        if len(hc) > 0 and len(lc) > 0:
            # Get all categories present in the data
            all_cats_in_data = set(hc + lc)
            
            # Order categories by hierarchy (most positive to most negative)
            mappings = CategoryMappings.get_mappings(hash_column)
            hierarchy = mappings['hierarchy']
            
            # Separate into categories with hierarchy values and those without
            categorized = [cat for cat in all_cats_in_data if cat in hierarchy]
            uncategorized = [cat for cat in all_cats_in_data if cat not in hierarchy]
            
            # Sort categorized by hierarchy value (descending - most positive first)
            categorized.sort(key=lambda x: hierarchy[x], reverse=True)
            
            # Combine: hierarchy-ordered first, then alphabetically sorted uncategorized
            all_cats = categorized + sorted(uncategorized)
            
            cm_cat = confusion_matrix(hc, lc, labels=all_cats)

            hs = sub['human_sentiment'].tolist()
            ls = sub['llm_sentiment'].tolist()
            sent_labels = ['Positive', 'Negative']
            cm_sent = confusion_matrix(hs, ls, labels=sent_labels)

            # Category confusion heatmap
            disp_cats = [c for c in all_cats]
            alias_labels = [aliases.get(lbl, lbl) for lbl in disp_cats]
            plt.figure(figsize=(10, 8))
            im = plt.imshow(cm_cat, cmap='Blues', aspect='auto')
            plt.title(f'{llm_name}', fontsize=16, fontweight='bold')
            plt.xlabel('LLM Predicted', fontsize=14)
            plt.ylabel('Human Ground Truth', fontsize=14)
            plt.xticks(range(len(alias_labels)), alias_labels, rotation=45, ha='right', fontsize=12)
            plt.yticks(range(len(alias_labels)), alias_labels, fontsize=12)
            cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
            cbar.set_label('Count', fontsize=12, rotation=90)
            
            # Annotate cells
            max_val = cm_cat.max() if cm_cat.size > 0 else 1
            for i in range(cm_cat.shape[0]):
                for j in range(cm_cat.shape[1]):
                    val = int(cm_cat[i, j])
                    plt.text(j, i, str(val),
                            ha='center', va='center',
                            color='white' if val > max_val/2 else 'black', 
                            fontsize=12, fontweight='bold')
            plt.tight_layout()
            plt.savefig(os.path.join(llm_dir, f'confusion_categories_{llm_name}_{analysis_type}.png'), dpi=300, bbox_inches='tight')
            plt.close()

            # Sentiment confusion heatmap
            plt.figure(figsize=(6, 5))
            im = plt.imshow(cm_sent, cmap='RdYlBu', aspect='auto')
            plt.title(f'Sentiment Confusion – {llm_name}', fontsize=14, fontweight='bold')
            plt.xlabel('LLM Predicted', fontsize=14)
            plt.ylabel('Human Ground Truth', fontsize=14)
            plt.xticks(range(len(sent_labels)), sent_labels, fontsize=16)
            plt.yticks(range(len(sent_labels)), sent_labels, fontsize=16)
            cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
            cbar.set_label('Count', fontsize=14, rotation=90)
            
            max_val = cm_sent.max() if cm_sent.size > 0 else 1
            for i in range(cm_sent.shape[0]):
                for j in range(cm_sent.shape[1]):
                    val = int(cm_sent[i, j])
                    plt.text(j, i, str(val),
                            ha='center', va='center',
                            color='white' if val > max_val/2 else 'black', 
                            fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.savefig(os.path.join(llm_dir, f'confusion_sentiment_{llm_name}_{analysis_type}.png'), dpi=300, bbox_inches='tight')
            plt.close()

        # Per-LLM stats row
        if stats_df is not None and not stats_df.empty:
            srow = stats_df[stats_df['llm_name'] == llm_name]
            if not srow.empty:
                srow.to_csv(os.path.join(llm_dir, f'stats_{llm_name}_{analysis_type}.csv'), index=False)

def _generate_per_llm_stats(results_df):
    """Generate per-LLM statistics from results_df"""
    stats_rows = []
    
    for llm_name in results_df['llm_name'].unique():
        sub = results_df[results_df['llm_name'] == llm_name]
        total = len(sub)
        
        if total > 0:
            cat_agreement = int(sub['category_agreement'].sum())
            sent_agreement = int(sub['sentiment_agreement'].sum())
            
            # Calculate percentages
            cat_pct = round((cat_agreement / total) * 100, 1)
            sent_pct = round((sent_agreement / total) * 100, 1)
            
            # Score differences
            score_diffs = sub['score_difference']
            mean_score_diff = round(score_diffs.mean(), 2)
            std_score_diff = round(score_diffs.std(), 2)
            max_score_diff = int(score_diffs.max())
            
            # Cohen's kappa for this LLM
            hc = sub['human_dominant_category'].tolist()
            lc = sub['llm_dominant_category'].tolist()
            all_cats = sorted(set(hc + lc))
            
            if len(all_cats) > 1:
                mapping = {c: i for i, c in enumerate(all_cats)}
                human_numeric = [mapping[c] for c in hc]
                llm_numeric = [mapping[c] for c in lc]
                kappa = round(cohen_kappa_score(human_numeric, llm_numeric), 3)
            else:
                kappa = 1.0
            
            stats_rows.append({
                'llm_name': llm_name,
                'total_comparisons': total,
                'category_agreement_count': cat_agreement,
                'category_agreement_percentage': cat_pct,
                'sentiment_agreement_count': sent_agreement,
                'sentiment_agreement_percentage': sent_pct,
                'cohens_kappa': kappa,
                'mean_score_difference': mean_score_diff,
                'std_score_difference': std_score_diff,
                'max_score_difference': max_score_diff
            })
    
    return pd.DataFrame(stats_rows)


def run_human_vs_llm_analysis_per_model(feedback_csv, hash_column='reaction_hash',
                                       output_dir='human_vs_llm_per_model',
                                       category_column='local_feedback',
                                       expert_column='source_file',
                                       text_column='local_feedback_text',
                                       confidence_column='confidence',
                                       llm_order=("GPT-4.1", "Claude", "GPT-o3", "Gemini")):
    """
    Wrapper: run the majority comparison, then save per-LLM outputs and combined matrix.
    """
    os.makedirs(output_dir, exist_ok=True)
    feedback_df = pd.read_csv(feedback_csv)

    analysis_type = 'Reactions' if hash_column == 'reaction_hash' else 'Routes'
    print(f"Starting Human vs LLM Majority Agreement Analysis (per LLM outputs) - {analysis_type}...")
    print("=" * 70)
    
    # Run analysis with specified parameters
    analysis = analyze_human_vs_llm_agreement(
        feedback_df,
        hash_column=hash_column,
        category_column=category_column,
        expert_column=expert_column,
        text_column=text_column,
        confidence_column=confidence_column
    )
    
    if not analysis:
        print("Analysis failed. Please check your data.")
        return

    # Generate per-LLM stats if not present
    if 'stats_df' not in analysis:
        analysis['stats_df'] = _generate_per_llm_stats(analysis['results_df'])

    # Save global results for reference
    analysis['results_df'].to_csv(os.path.join(output_dir, 'human_vs_llm_majority_all.csv'), index=False)
    analysis['matrix_df'].to_csv(os.path.join(output_dir, 'human_vs_llm_majority_matrix_all.csv'), index=False)
    analysis['stats_df'].to_csv(os.path.join(output_dir, 'human_vs_llm_majority_stats_all.csv'), index=False)


    # Save per-LLM outputs and combined matrix heatmap
    save_per_llm_outputs(analysis, output_dir, hash_column=hash_column, llm_order=llm_order)

    print(f"\nPer-LLM outputs saved under {output_dir}/")
    return analysis

In [ ]:
analysis_results = run_human_vs_llm_analysis_per_model(
    'expert_feedback_combined_llms.csv',
    hash_column='reaction_hash',
    output_dir='reactions_analysis'
)

In [ ]:
results = run_human_vs_llm_analysis_per_model(
    'expert_feedback_combined_llms.csv', 
    hash_column='root_molecule_hash',
    category_column='general_feedback',
    expert_column='source_file',
    text_column='general_feedback_text',
    output_dir='routes_analysis',
)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

def save_category_distribution_per_llm(feedback_df, hash_column='reaction_hash',
                output_dir='human_vs_llm_analysis_reruns',
                category_column='local_feedback',
                expert_column='source_file',
                llm_order=("GPT-4.1", "Claude", "GPT-o3", "Gemini"),
                truncate=55,
                figsize=(12, 7),
                sort_by='count'):
    """
    Save category distribution plots for:
    1. Each LLM (all runs) with human baseline - using consistent scale
    2. Human distribution
    3. Combined comparison plots
    
    Parameters:
    - feedback_df: DataFrame containing all evaluations
    - hash_column: 'reaction_hash' or 'route_hash' to determine analysis type
    - output_dir: directory to save outputs
    - category_column: column with category labels
    - expert_column: column with model identifier (LLM name)
    - llm_order: tuple/list of LLMs to plot and their order
    - truncate: max characters for category label display
    - figsize: figure size for each plot
    - sort_by: 'percentage' or 'count' to sort bars
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Get mappings for this analysis type
    mappings = CategoryMappings.get_mappings(hash_column)
    aliases = mappings.get('aliases', {})
    hierarchy = mappings['hierarchy']
    
    # Separate human and LLM data
    human_df = feedback_df[~feedback_df[expert_column].isin(llm_order)].copy()
    llm_df = feedback_df[feedback_df[expert_column].isin(llm_order)].copy()
    
    if llm_df.empty:
        print("No LLM rows found for the specified llm_order.")
        return
    
    # Calculate human baseline distribution (simple vote counting)
    human_baseline = _calculate_human_distribution(human_df, category_column)
    
    # Get all categories and determine global scale
    all_categories = set(llm_df[category_column].dropna().unique())
    all_categories.update(human_baseline.keys())
    categories = list(all_categories)
    categories.sort(key=lambda x: hierarchy.get(x, -np.inf), reverse=True)
    
    # Calculate max percentage across all LLMs and humans for consistent scale
    max_percentage = _calculate_global_max_percentage(llm_df, human_baseline, llm_order, 
                                                     categories, category_column, expert_column)
    
    # Save human distribution plot
    _save_human_distribution_plot(human_baseline, aliases, hierarchy, output_dir, 
                                 figsize, sort_by, max_percentage)
    
    # Process each LLM with consistent scale
    for llm in llm_order:
        llm_sub = llm_df[llm_df[expert_column] == llm]
        if llm_sub.empty:
            print(f"No rows for LLM: {llm}")
            _save_empty_plot(llm, output_dir)
            continue
        
        # All runs distribution with human baseline and consistent scale
        _save_llm_distribution(llm_sub, llm, human_baseline, aliases, hierarchy, 
                              category_column, output_dir, figsize, sort_by, 
                              categories, max_percentage)
    
    # Combined comparison plots
    _save_combined_distribution_plots(llm_df, human_baseline, llm_order, aliases, hierarchy,
                                     category_column, output_dir, figsize, sort_by, categories)

def _calculate_global_max_percentage(llm_df, human_baseline, llm_order, categories, 
                                   category_column, expert_column):
    """Calculate the maximum percentage across all LLMs and humans for consistent scaling"""
    max_percentages = []
    
    # Add human max
    if human_baseline:
        max_percentages.append(max(human_baseline.values()))
    
    # Add each LLM max
    for llm in llm_order:
        llm_sub = llm_df[llm_df[expert_column] == llm]
        if not llm_sub.empty:
            counts = llm_sub[category_column].value_counts()
            total = counts.sum()
            if total > 0:
                percentages = (counts / total * 100.0)
                max_percentages.append(percentages.max())
    
    # Return max with some padding for better visualization
    global_max = max(max_percentages) if max_percentages else 100
    return min(100, global_max * 1.1)  # Add 10% padding, cap at 100%

def _calculate_human_distribution(human_df, category_column):
    """Calculate human category distribution by counting all votes"""
    if human_df.empty:
        return {}
    
    # Simple vote counting - count every category selection by humans
    counts = human_df[category_column].value_counts()
    total = counts.sum()
    
    if total == 0:
        return {}
    
    return {cat: (count / total * 100.0) for cat, count in counts.items()}

def _save_human_distribution_plot(human_baseline, aliases, hierarchy, output_dir, 
                                 figsize, sort_by, max_percentage):
    """Save human distribution plot with consistent scale"""
    if not human_baseline:
        return
    
    # Order categories by hierarchy
    categories = list(human_baseline.keys())
    categories.sort(key=lambda x: hierarchy.get(x, -np.inf), reverse=True)
    
    percentages = [human_baseline[cat] for cat in categories]
    alias_labels = [aliases.get(cat, cat) for cat in categories]
    
    plt.figure(figsize=figsize)
    bars = plt.bar(range(len(categories)), percentages, color='darkgreen', alpha=0.7, edgecolor='black')
    plt.title('Human Expert Distribution (All Votes)', fontsize=16, fontweight='bold')
    plt.ylabel('Percentage (%)', fontsize=14)
    plt.ylim(0, max_percentage)  # Set consistent scale
    plt.xticks(range(len(alias_labels)), alias_labels, rotation=45, fontsize=12, ha='right')
    plt.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Annotate bars
    for i, bar in enumerate(bars):
        height = bar.get_height()
        if height >= max_percentage * 0.02:  # Annotate if >= 2% of max scale
            plt.text(bar.get_x() + bar.get_width()/2., height + max_percentage * 0.01, f'{height:.1f}%',
                    ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'category_distribution_humans.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # Save CSV
    human_df = pd.DataFrame({
        'category': categories,
        'percentage': percentages
    })
    human_df.to_csv(os.path.join(output_dir, 'category_distribution_humans.csv'), index=False)

def _save_llm_distribution(llm_sub, llm_name, human_baseline, aliases, hierarchy,
                          category_column, output_dir, figsize, sort_by, 
                          categories, max_percentage):
    """Save LLM distribution for all runs with human baseline and consistent scale"""
    counts = llm_sub[category_column].value_counts()
    total = counts.sum()
    percentages = (counts / total * 100.0).round(1)
    
    # Use the provided categories order (already sorted by hierarchy)
    llm_percentages = [percentages.get(cat, 0) for cat in categories]
    human_percentages = [human_baseline.get(cat, 0) for cat in categories]
    alias_labels = [aliases.get(cat, cat) for cat in categories]
    
    # Create table and save CSV
    table = pd.DataFrame({
        'category': categories,
        'count': [counts.get(cat, 0) for cat in categories],
        'percentage': llm_percentages,
        'human_baseline_percentage': human_percentages
    })
    table.to_csv(os.path.join(output_dir, f'category_distribution_{llm_name}.csv'), index=False)
    
    # Plot with consistent scale
    plt.figure(figsize=figsize)
    x_pos = np.arange(len(categories))
    
    bars = plt.bar(x_pos, llm_percentages, color='steelblue', alpha=0.8, 
                   edgecolor='black', label=f'{llm_name}')
    
    # Add human baseline as dashed line for "Reaction feasible, all good"
    target_category = "Reaction feasible, all good"
    if target_category in categories and target_category in human_baseline:
        human_value = human_baseline[target_category]
        plt.axhline(y=human_value, color='darkgreen', linestyle='--', linewidth=2, 
                   label=f'Human Baseline ({target_category}): {human_value:.1f}%')
    
    plt.title(f'{llm_name}', fontsize=16, fontweight='bold')
    plt.ylabel('Percentage (%)', fontsize=14)
    plt.ylim(0, max_percentage)  # Set consistent scale
    plt.xticks(x_pos, alias_labels, rotation=45, fontsize=12, ha='right')
    plt.grid(axis='y', alpha=0.3, linestyle='--')
    plt.legend(fontsize=12)
    
    # Annotate bars
    for i, bar in enumerate(bars):
        height = bar.get_height()
        if height >= max_percentage * 0.02:  # Annotate if >= 2% of max scale
            plt.text(bar.get_x() + bar.get_width()/2., height + max_percentage * 0.01, f'{height:.1f}%',
                    ha='center', va='bottom', fontsize=11)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'category_distribution_{llm_name}.png'), 
                dpi=300, bbox_inches='tight')
    plt.close()

def _save_combined_distribution_plots(llm_df, human_baseline, llm_order, aliases, hierarchy,
                                     category_column, output_dir, figsize, sort_by, categories):
    """Save combined comparison plots"""
    alias_labels = [aliases.get(cat, cat) for cat in categories]
    
    # Combined comparison
    plt.figure(figsize=(15, 8))
    x = np.arange(len(categories))
    width = 0.15
    
    # Human baseline
    human_values = [human_baseline.get(cat, 0) for cat in categories]
    plt.bar(x - 2*width, human_values, width, label='Humans', color='darkgreen', alpha=0.7)
    
    # Each LLM
    colors = ['steelblue', 'orange', 'red', 'purple']
    expert_col = [col for col in llm_df.columns if 'source' in col or 'expert' in col][0]
    
    for i, llm in enumerate(llm_order):
        llm_sub = llm_df[llm_df[expert_col] == llm]
        if not llm_sub.empty:
            counts = llm_sub[category_column].value_counts()
            total = counts.sum()
            percentages = {cat: (counts.get(cat, 0) / total * 100.0) for cat in categories}
            values = [percentages[cat] for cat in categories]
            plt.bar(x + (i-1)*width, values, width, label=f'{llm}', 
                   color=colors[i % len(colors)], alpha=0.7)
    
    plt.title('Category Distribution Comparison', fontsize=16, fontweight='bold')
    plt.ylabel('Percentage (%)', fontsize=14)
    plt.xlabel('Categories', fontsize=14)
    plt.xticks(x, alias_labels, rotation=45, ha='right', fontsize=10)
    plt.legend(fontsize=12, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='y', alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'category_distribution_comparison.png'), 
                dpi=300, bbox_inches='tight')
    plt.close()
    
    # Save combined data as CSV
    combined_data = {'category': categories}
    combined_data['humans'] = human_values
    
    for llm in llm_order:
        llm_sub = llm_df[llm_df[expert_col] == llm]
        if not llm_sub.empty:
            counts = llm_sub[category_column].value_counts()
            total = counts.sum()
            values = [(counts.get(cat, 0) / total * 100.0) for cat in categories]
            combined_data[llm] = values
        else:
            combined_data[llm] = [0] * len(categories)
    
    combined_df = pd.DataFrame(combined_data)
    combined_df.to_csv(os.path.join(output_dir, 'category_distribution_comparison.csv'), index=False)

def _save_empty_plot(name, output_dir):
    """Save empty placeholder plot"""
    plt.figure(figsize=(8, 3))
    plt.text(0.5, 0.5, f'No data for {name}', ha='center', va='center', fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'category_distribution_{name}.png'), 
                dpi=300, bbox_inches='tight')
    plt.close()

# Usage function
def run_category_distribution_analysis(feedback_csv, hash_column='reaction_hash',
                                     output_dir='category_distribution_analysis',
                                     category_column='local_feedback',
                                     expert_column='source_file',
                                     llm_order=("GPT-4.1", "Claude", "GPT-o3", "Gemini")):
    """
    Run complete category distribution analysis with consistent scaling
    """
    feedback_df = pd.read_csv(feedback_csv)
    
    analysis_type = 'Reactions' if hash_column == 'reaction_hash' else 'Routes'
    print(f"Starting Category Distribution Analysis - {analysis_type}...")
    print("=" * 50)
    
    save_category_distribution_per_llm(
        feedback_df=feedback_df,
        hash_column=hash_column,
        output_dir=output_dir,
        category_column=category_column,
        expert_column=expert_column,
        llm_order=llm_order
    )
    
    print(f"Results saved to {output_dir}/")

In [ ]:
run_category_distribution_analysis(
    'expert_feedback_combined_llms.csv',
    hash_column='reaction_hash',
    output_dir='reaction_distributions'
)

In [ ]:
run_category_distribution_analysis(
    'expert_feedback_combined_llms.csv',
    hash_column='root_molecule_hash',
    category_column='general_feedback',
    expert_column='source_file',
    output_dir='routes_distributions',
)

In [ ]:
def _analyze_single_llm_rerun_agreement(reaction_data, reaction_hash, llm_name,
                                       category_column, text_column, confidence_column,
                                       consensus_result, hierarchy, sentiment_mapping):
    """
    Analyze agreement for a single reaction within one LLM's re-runs.
    Uses the consensus result from the existing tie-breaking logic.
    """
    # Collect categories from re-runs
    categories = reaction_data[category_column].dropna().tolist()
    num_runs = len(categories)

    # Get consensus category and agreement percentage from existing logic
    dominant_category = consensus_result['dominant_category']
    category_agreement_pct = consensus_result['agreement_pct']
    
    # Determine agreement level based on percentage
    if category_agreement_pct >= 100 - 1e-9:
        agreement_level = "Perfect"
    elif category_agreement_pct >= 75:
        agreement_level = "Strong"
    elif category_agreement_pct >= 50:
        agreement_level = "Moderate"
    elif category_agreement_pct >= 30:
        agreement_level = "Weak"
    else:
        agreement_level = "None"
        # Print detailed reason for None consensus
        _print_none_consensus_reason(llm_name, reaction_hash, categories, 
                                   consensus_result, category_agreement_pct)

    # Sentiment analysis
    sentiments = [sentiment_mapping.get(c, 'Unknown') for c in categories]
    sent_counts = Counter(sentiments)
    top_sent = sent_counts.most_common(1)[0][0] if sent_counts else 'Unknown'
    sent_agree_pct = (sent_counts.get(top_sent, 0) / num_runs * 100.0) if num_runs > 0 else 0.0

    # Score summary using hierarchy
    run_scores = [hierarchy.get(c, 0) for c in categories]
    score_mean = float(np.mean(run_scores)) if run_scores else 0.0
    score_std = float(np.std(run_scores)) if run_scores else 0.0

    # Optional confidence analysis
    confidences = reaction_data[confidence_column].dropna().tolist() if confidence_column in reaction_data.columns else []
    confidence_mean = np.mean(confidences) if len(confidences) > 0 else None
    confidence_std = np.std(confidences) if len(confidences) > 0 else None

    # Texts
    texts = reaction_data[text_column].dropna().tolist() if text_column in reaction_data.columns else []

    return {
        'llm_name': llm_name,
        'reaction_hash': reaction_hash,
        'num_runs': num_runs,
        'dominant_category': dominant_category,
        'category_agreement_pct': round(category_agreement_pct, 1),
        'dominant_sentiment': top_sent,
        'sentiment_agreement_pct': round(sent_agree_pct, 1),
        'agreement_level': agreement_level,
        'score_mean': round(score_mean, 2),
        'score_std': round(score_std, 2),
        'confidence_mean': round(confidence_mean, 1) if confidence_mean is not None else None,
        'confidence_std': round(confidence_std, 1) if confidence_std is not None else None,
        'category_counts': consensus_result['all_categories'],
        'sentiment_counts': dict(sent_counts),
        'unique_categories': len(consensus_result['all_categories']),
        'unique_sentiments': len(sent_counts),
        'text_feedback': texts,
        'has_disagreement': agreement_level not in ['Perfect']
    }

def _analyze_single_human_agreement(reaction_data, reaction_hash, expert_column,
                                  category_column, text_column, consensus_result,
                                  hierarchy, sentiment_mapping):
    """Analyze agreement for a single reaction among human experts"""
    experts = reaction_data[expert_column].unique()
    num_experts = len(experts)
    
    # Get consensus from existing logic
    dominant_category = consensus_result['dominant_category']
    category_agreement_pct = consensus_result['agreement_pct']
    
    # Determine agreement level
    if category_agreement_pct >= 100 - 1e-9:
        agreement_level = "Perfect"
    elif category_agreement_pct >= 75:
        agreement_level = "Strong"
    elif category_agreement_pct >= 50:
        agreement_level = "Moderate"
    elif category_agreement_pct >= 30:
        agreement_level = "Weak"
    else:
        agreement_level = "None"
        # Print detailed reason for None consensus
        all_categories = []
        for expert in experts:
            expert_data = reaction_data[reaction_data[expert_column] == expert]
            expert_cats = expert_data[category_column].dropna().tolist()
            all_categories.extend(expert_cats)
        _print_none_consensus_reason("Human Experts", reaction_hash, all_categories, 
                                   consensus_result, category_agreement_pct)

    # Collect all categories from all experts (presence voting already handled in consensus)
    all_categories = []
    for expert in experts:
        expert_data = reaction_data[reaction_data[expert_column] == expert]
        expert_cats = expert_data[category_column].dropna().tolist()
        all_categories.extend(expert_cats)
    
    # Sentiment analysis
    sentiments = [sentiment_mapping.get(c, 'Unknown') for c in all_categories]
    sent_counts = Counter(sentiments)
    top_sent = sent_counts.most_common(1)[0][0] if sent_counts else 'Unknown'
    sent_agree_pct = (sent_counts.get(top_sent, 0) / len(sentiments) * 100.0) if sentiments else 0.0

    return {
        'reaction_hash': reaction_hash,
        'num_experts': num_experts,
        'dominant_category': dominant_category,
        'category_agreement_pct': round(category_agreement_pct, 1),
        'dominant_sentiment': top_sent,
        'sentiment_agreement_pct': round(sent_agree_pct, 1),
        'agreement_level': agreement_level,
        'category_counts': consensus_result['all_categories'],
        'sentiment_counts': dict(sent_counts),
        'unique_categories': len(consensus_result['all_categories']),
        'unique_sentiments': len(sent_counts),
        'has_disagreement': agreement_level not in ['Perfect']
    }

def _print_none_consensus_reason(evaluator_name, reaction_hash, categories, consensus_result, agreement_pct):
    """Print detailed reason why consensus is 'None' (< 30% agreement)"""
    print(f"\n{'='*60}")
    print(f"NONE CONSENSUS DETECTED")
    print(f"{'='*60}")
    print(f"Evaluator: {evaluator_name}")
    print(f"Reaction Hash: {reaction_hash}")
    print(f"Agreement Level: {agreement_pct:.1f}% (< 30% threshold)")
    print(f"Total Responses: {len(categories)}")
    print(f"Unique Categories: {len(set(categories))}")
    
    # Show category breakdown
    cat_counts = Counter(categories)
    print(f"\nCategory Distribution:")
    for cat, count in cat_counts.most_common():
        pct = (count / len(categories)) * 100
        print(f"  • {cat}: {count} votes ({pct:.1f}%)")
    
    # Show consensus choice and reasoning
    print(f"\nConsensus Choice: '{consensus_result['dominant_category']}'")
    print(f"Consensus Logic: Pessimistic tie-breaking applied")
    
    # Show why it's problematic
    max_category_pct = max((count / len(categories)) * 100 for count in cat_counts.values())
    print(f"Highest Single Category: {max_category_pct:.1f}%")
    
    if len(set(categories)) == len(categories):
        print("Issue: Complete disagreement - every response different")
    elif max_category_pct < 30:
        print("Issue: No category reaches 30% threshold")
    else:
        print("Issue: High fragmentation across multiple categories")
    
    print(f"{'='*60}\n")

def _create_agreement_pie_chart(data_df, name, color_map, title_suffix, output_dir, agreement_col):
    """Create individual agreement pie chart with improved formatting"""
    dist = data_df[agreement_col].value_counts()
    total = len(data_df)

    plt.figure(figsize=(10, 8))
    if total == 0 or dist.empty:
        plt.pie([1], labels=[f'No eligible {title_suffix.lower()}'], colors=['#CCCCCC'])
        plt.title(f'{name}: Agreement Levels\n(No eligible {title_suffix.lower()})', 
                 fontsize=16, fontweight='bold', pad=20)
    else:
        labels = ['Perfect', 'Strong', 'Moderate', 'Weak', 'None']
        values = [dist.get(l, 0) for l in labels]
        labels_f = [l for l, v in zip(labels, values) if v > 0]
        values_f = [v for v in values if v > 0]
        colors_f = [color_map.get(l, '#CCCCCC') for l in labels_f]
        
        if not values_f:
            plt.pie([1], labels=['No levels'], colors=['#DDDDDD'])
            plt.title(f'{name}: Agreement Levels\n(0 {title_suffix.lower()})', 
                     fontsize=16, fontweight='bold', pad=20)
        else:
            # Create pie chart with better spacing and formatting
            wedges, texts, autotexts = plt.pie(
                values_f, 
                labels=labels_f, 
                autopct=lambda pct: f'{pct:.1f}%' if pct > 3 else '',  # Only show % if > 3%
                colors=colors_f, 
                startangle=0,
                pctdistance=0.65,  # Move percentages closer to edge
                labeldistance=1.05,  # Move labels further from center
                textprops=dict(fontsize=16, fontweight='bold')
            )
            
            # Style the percentage text
            for autotext in autotexts:
                autotext.set_color('black')  # Black text instead of white
                autotext.set_fontweight('bold')
                autotext.set_fontsize(20)
                autotext.set_bbox(dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8, edgecolor='none'))
            
            # Style the label text
            for text in texts:
                text.set_fontsize(16)
                text.set_fontweight('bold')
                text.set_color('black')
            
            # Add a legend for better clarity
            #plt.legend(wedges, [f'{label}: {value} ({value/total*100:.1f}%)' for label, value in zip(labels_f, values_f)],
            #          title="Agreement Levels",
            #          loc="center left",
            #          bbox_to_anchor=(1, 0, 0.5, 1),
            #          fontsize=14)
            
            plt.title(f'{name}\n({total} {title_suffix.lower()})', 
                     fontsize=16, fontweight='bold', pad=20)
    
    # Ensure equal aspect ratio and tight layout
    plt.axis('equal')
    safe_name = name.replace(' ', '_').replace(':', '').replace('-', '_')
    fname = os.path.join(output_dir, f'agreement_levels_{safe_name}.png')
    plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def _create_combined_agreement_grid_integrated(reactions_df, human_df, color_map, title_suffix, output_dir):
    """Create combined grid with humans and LLMs with improved formatting"""
    llm_names = reactions_df['llm_name'].unique().tolist()
    total_plots = len(llm_names) + 1  # +1 for humans
    
    if total_plots <= 4:
        rows, cols = 2, 2
        figsize = (16, 12)
    elif total_plots <= 6:
        rows, cols = 2, 3
        figsize = (20, 12)
    else:
        rows = int(np.ceil(total_plots / 4))
        cols = 4
        figsize = (20, 5 * rows)

    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if total_plots == 1:
        axes = [axes]
    elif rows == 1:
        axes = axes.reshape(1, -1)
    
    plot_idx = 0

    # Human experts subplot
    if plot_idx < len(axes.flat):
        ax = axes.flat[plot_idx]
        dist = human_df['agreement_level'].value_counts()
        total = len(human_df)
        
        if total > 0 and not dist.empty:
            labels = ['Perfect', 'Strong', 'Moderate', 'Weak', 'None']
            values = [dist.get(l, 0) for l in labels]
            labels_f = [l for l, v in zip(labels, values) if v > 0]
            values_f = [v for v in values if v > 0]
            colors_f = [color_map.get(l, '#CCCCCC') for l in labels_f]
            
            if values_f:
                wedges, texts, autotexts = ax.pie(
                    values_f, 
                    labels=labels_f, 
                    autopct=lambda pct: f'{pct:.1f}%' if pct > 5 else '',
                    colors=colors_f, 
                    startangle=90,
                    pctdistance=0.8,
                    wedgeprops=dict(edgecolor='white', linewidth=1.5),
                    textprops=dict(fontsize=10)
                )
                
                # Style percentage text
                for autotext in autotexts:
                    autotext.set_color('black')
                    autotext.set_fontweight('bold')
                    autotext.set_fontsize(11)
                    autotext.set_bbox(dict(boxstyle="round,pad=0.2", facecolor='white', alpha=0.9, edgecolor='none'))
                
                # Style labels
                for text in texts:
                    text.set_fontsize(10)
                    text.set_fontweight('bold')
                
                ax.set_title(f'Human Experts\n({total} {title_suffix.lower()})', 
                           fontweight='bold', fontsize=12, pad=15)
            else:
                ax.pie([1], labels=['No levels'], colors=['#DDDDDD'])
                ax.set_title('Human Experts\n(No levels)', fontweight='bold', fontsize=12, pad=15)
        else:
            ax.pie([1], labels=['No data'], colors=['#CCCCCC'])
            ax.set_title('Human Experts\n(No data)', fontweight='bold', fontsize=12, pad=15)
        
        ax.axis('equal')
        plot_idx += 1

    # LLM subplots
    for llm in llm_names:
        if plot_idx >= len(axes.flat):
            break
            
        ax = axes.flat[plot_idx]
        sub = reactions_df[reactions_df['llm_name'] == llm]
        dist = sub['agreement_level'].value_counts()
        total = len(sub)
        
        if total == 0 or dist.empty:
            ax.pie([1], labels=['No data'], colors=['#CCCCCC'])
            ax.set_title(f'{llm}\n(No data)', fontweight='bold', fontsize=12, pad=15)
        else:
            labels = ['Perfect', 'Strong', 'Moderate', 'Weak', 'None']
            values = [dist.get(l, 0) for l in labels]
            labels_f = [l for l, v in zip(labels, values) if v > 0]
            values_f = [v for v in values if v > 0]
            colors_f = [color_map.get(l, '#CCCCCC') for l in labels_f]
            
            if not values_f:
                ax.pie([1], labels=['No levels'], colors=['#DDDDDD'])
                ax.set_title(f'{llm}\n(No levels)', fontweight='bold', fontsize=12, pad=15)
            else:
                wedges, texts, autotexts = ax.pie(
                    values_f, 
                    labels=labels_f, 
                    autopct=lambda pct: f'{pct:.1f}%' if pct > 5 else '',
                    colors=colors_f, 
                    startangle=90,
                    pctdistance=0.8,
                    wedgeprops=dict(edgecolor='white', linewidth=1.5),
                    textprops=dict(fontsize=10)
                )
                
                # Style percentage text
                for autotext in autotexts:
                    autotext.set_color('black')
                    autotext.set_fontweight('bold')
                    autotext.set_fontsize(11)
                    autotext.set_bbox(dict(boxstyle="round,pad=0.2", facecolor='white', alpha=0.9, edgecolor='none'))
                
                # Style labels
                for text in texts:
                    text.set_fontsize(10)
                    text.set_fontweight('bold')
                
                ax.set_title(f'{llm}\n({total} {title_suffix.lower()})', 
                           fontweight='bold', fontsize=12, pad=15)
        
        ax.axis('equal')
        plot_idx += 1

    # Hide unused subplots
    for idx in range(plot_idx, len(axes.flat)):
        axes.flat[idx].set_visible(False)

    # Add overall legend
    legend_elements = [plt.Rectangle((0,0),1,1, facecolor=color_map[level], edgecolor='white', linewidth=1) 
                      for level in ['Perfect', 'Strong', 'Moderate', 'Weak', 'None']]
    fig.legend(legend_elements, ['Perfect (100%)', 'Strong (75-99%)', 'Moderate (50-74%)', 'Weak (30-49%)', 'None (<30%)'],
              loc='center', bbox_to_anchor=(0.5, 0.02), ncol=5, fontsize=12, frameon=True, fancybox=True, shadow=True)

    plt.suptitle(f'Agreement Levels Comparison - {title_suffix}', fontsize=18, fontweight='bold', y=0.95)
    plt.tight_layout(rect=[0, 0.08, 1, 0.92])
    fname = os.path.join(output_dir, f'agreement_levels_combined_{title_suffix.lower()}.png')
    plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def analyze_llm_internal_agreement(feedback_df, hash_column='reaction_hash',
                                 category_column='local_feedback',
                                 expert_column='source_file',
                                 text_column='local_feedback_text',
                                 confidence_column='confidence',
                                 min_runs=2,
                                 llm_order=("Gemini", "GPT-o3", "GPT-4.1", "Claude")):
    """
    Analyze internal agreement within LLM re-runs using existing tie-breaking rules.
    Integrated with the refactored analyzer architecture.
    """
    # Initialize analyzers with correct mappings
    group_analyzer = GroupConsensusAnalyzer(hash_column)
    mappings = CategoryMappings.get_mappings(hash_column)
    hierarchy = mappings['hierarchy']
    sentiment_mapping = mappings['sentiment']
    
    feedback_df = feedback_df.dropna(subset=[hash_column])
    llm_feedback = feedback_df[feedback_df[expert_column].isin(llm_order)].copy()

    if llm_feedback.empty:
        print("No LLM feedback found for the specified models.")
        return {}

    print(f"Analyzing re-runs for {len(llm_feedback[expert_column].unique())} LLMs")
    print(f"Total LLM feedback entries: {len(llm_feedback)}")

    # Build list of valid LLM-reaction sets with at least min_runs
    valid_llm_reactions = []
    for llm in llm_order:
        df_llm = llm_feedback[llm_feedback[expert_column] == llm]
        for reaction_hash, group in df_llm.groupby(hash_column):
            n_runs = group[category_column].dropna().shape[0]
            if n_runs >= min_runs:
                valid_llm_reactions.append((llm, reaction_hash))

    print(f"LLM-reaction pairs with {min_runs}+ re-runs: {len(valid_llm_reactions)}")
    if not valid_llm_reactions:
        print(f"No LLM-reaction pairs found with at least {min_runs} re-runs.")
        return {}

    reaction_analyses = []

    for llm, reaction_hash in valid_llm_reactions:
        reaction_data = llm_feedback[(llm_feedback[expert_column] == llm) & 
                                   (llm_feedback[hash_column] == reaction_hash)]
        
        # Use existing group consensus logic for consistency
        consensus_result = group_analyzer.get_group_consensus(
            reaction_data, expert_column, category_column, text_column, confidence_column
        )
        
        # Analyze agreement within this LLM's re-runs for this reaction
        reaction_analysis = _analyze_single_llm_rerun_agreement(
            reaction_data, reaction_hash, llm, category_column, text_column, 
            confidence_column, consensus_result, hierarchy, sentiment_mapping
        )
        reaction_analyses.append(reaction_analysis)

    reactions_df = pd.DataFrame(reaction_analyses)

    # Calculate overall statistics for pie charts
    overall_stats = _calculate_llm_rerun_statistics(reactions_df)

    analysis = {
        'reactions_df': reactions_df,
        'overall_statistics': overall_stats
    }

    return analysis

def _calculate_llm_rerun_statistics(reactions_df):
    """Calculate basic statistics needed for visualizations"""
    total_reactions = len(reactions_df)
    
    if total_reactions == 0:
        return {
            'total_reactions': 0,
            'agreement_level_distribution': {},
            'agreement_level_percentage': {}
        }
    
    # Agreement level distribution
    agreement_dist = reactions_df['agreement_level'].value_counts()
    agreement_dist_pct = (agreement_dist / total_reactions * 100).round(1)
    
    return {
        'total_reactions': total_reactions,
        'agreement_level_distribution': agreement_dist.to_dict(),
        'agreement_level_percentage': agreement_dist_pct.to_dict(),
        'average_runs_per_reaction': round(reactions_df['num_runs'].mean(), 1) if 'num_runs' in reactions_df.columns else 0.0
    }

def analyze_human_internal_agreement(feedback_df, hash_column='reaction_hash',
                                   category_column='local_feedback',
                                   expert_column='source_file',
                                   text_column='local_feedback_text',
                                   min_experts=2,
                                   llm_order=("Gemini", "GPT-o3", "GPT-4.1", "Claude")):
    """
    Analyze internal agreement among human experts using existing tie-breaking rules.
    """
    # Initialize analyzers
    group_analyzer = GroupConsensusAnalyzer(hash_column)
    mappings = CategoryMappings.get_mappings(hash_column)
    hierarchy = mappings['hierarchy']
    sentiment_mapping = mappings['sentiment']
    
    feedback_df = feedback_df.dropna(subset=[hash_column])
    human_feedback = feedback_df[~feedback_df[expert_column].isin(llm_order)].copy()

    if human_feedback.empty:
        print("No human feedback found.")
        return {}

    print(f"Analyzing agreement for {len(human_feedback[expert_column].unique())} human experts")
    print(f"Total human feedback entries: {len(human_feedback)}")

    # Find reactions with multiple human experts
    valid_reactions = []
    for reaction_hash, group in human_feedback.groupby(hash_column):
        n_experts = len(group[expert_column].unique())
        if n_experts >= min_experts:
            valid_reactions.append(reaction_hash)

    print(f"Reactions with {min_experts}+ human experts: {len(valid_reactions)}")
    if not valid_reactions:
        print(f"No reactions found with at least {min_experts} human experts.")
        return {}

    reaction_analyses = []

    for reaction_hash in valid_reactions:
        reaction_data = human_feedback[human_feedback[hash_column] == reaction_hash]
        
        # Use existing group consensus logic
        consensus_result = group_analyzer.get_group_consensus(
            reaction_data, expert_column, category_column, text_column
        )
        
        # Analyze agreement among human experts
        reaction_analysis = _analyze_single_human_agreement(
            reaction_data, reaction_hash, expert_column, category_column, 
            text_column, consensus_result, hierarchy, sentiment_mapping
        )
        reaction_analyses.append(reaction_analysis)

    reactions_df = pd.DataFrame(reaction_analyses)
    overall_stats = _calculate_human_agreement_statistics(reactions_df)

    analysis = {
        'human_agreement_df': reactions_df,
        'overall_statistics': overall_stats
    }

    return analysis

def _calculate_human_agreement_statistics(reactions_df):
    """Calculate basic statistics for human agreement"""
    total_reactions = len(reactions_df)
    
    if total_reactions == 0:
        return {
            'total_reactions': 0,
            'agreement_level_distribution': {},
            'agreement_level_percentage': {}
        }
    
    agreement_dist = reactions_df['agreement_level'].value_counts()
    agreement_dist_pct = (agreement_dist / total_reactions * 100).round(1)
    
    return {
        'total_reactions': total_reactions,
        'agreement_level_distribution': agreement_dist.to_dict(),
        'agreement_level_percentage': agreement_dist_pct.to_dict(),
        'average_experts_per_reaction': round(reactions_df['num_experts'].mean(), 1) if 'num_experts' in reactions_df.columns else 0.0
    }

# Updated visualization function to work with integrated analysis
def visualize_agreement_levels(llm_analysis_results, human_analysis_results, output_dir, hash_column='reaction_hash'):
    """
    Visualizations for agreement levels using integrated analysis results
    """
    os.makedirs(output_dir, exist_ok=True)
    
    title_suffix = 'Reactions' if hash_column == 'reaction_hash' else 'Routes'
    
    # Color mapping for agreement levels
    color_map = {
        'Perfect': '#006400',     # darkest green
        'Strong':  '#7CFC00',     # light green
        'Moderate':'#FFD700',     # yellow
        'Weak':    '#FFA500',     # orange
        'None':    '#FF0000'      # red
    }

    # Get LLM data
    reactions_df = llm_analysis_results.get('reactions_df')
    if reactions_df is not None and not reactions_df.empty:
        llm_names = reactions_df['llm_name'].unique().tolist()
        
        # Individual LLM pie charts
        for llm in llm_names:
            sub = reactions_df[reactions_df['llm_name'] == llm]
            _create_agreement_pie_chart(sub, llm, color_map, title_suffix, output_dir, 'agreement_level')
    
    # Human agreement pie chart
    human_df = human_analysis_results.get('human_agreement_df')
    if human_df is not None and not human_df.empty:
        _create_agreement_pie_chart(human_df, 'Human Experts', color_map, title_suffix, output_dir, 'agreement_level')
    
    # Combined grid
    if reactions_df is not None and human_df is not None:
        _create_combined_agreement_grid_integrated(reactions_df, human_df, color_map, title_suffix, output_dir)

# Usage function
def run_complete_agreement_analysis(feedback_csv, hash_column='reaction_hash',
                                  output_dir='agreement_analysis',
                                  category_column='local_feedback',
                                  expert_column='source_file',
                                  text_column='local_feedback_text',
                                  confidence_column='confidence',
                                  llm_order=("Gemini", "GPT-o3", "GPT-4.1", "Claude")):
    """
    Run complete agreement analysis for both LLMs and humans with pie chart visualizations
    """
    feedback_df = pd.read_csv(feedback_csv)
    
    analysis_type = 'Reactions' if hash_column == 'reaction_hash' else 'Routes'
    print(f"Starting Complete Agreement Analysis - {analysis_type}...")
    print("=" * 50)
    
    # Analyze LLM internal agreement
    llm_analysis = analyze_llm_internal_agreement(
        feedback_df,
        hash_column=hash_column,
        category_column=category_column,
        expert_column=expert_column,
        text_column=text_column,
        confidence_column=confidence_column,
        llm_order=llm_order
    )
    
    # Analyze human internal agreement
    human_analysis = analyze_human_internal_agreement(
        feedback_df,
        hash_column=hash_column,
        category_column=category_column,
        expert_column=expert_column,
        text_column=text_column,
        llm_order=llm_order
    )
    
    # Create visualizations
    visualize_agreement_levels(llm_analysis, human_analysis, output_dir, hash_column)
    
    print(f"Results saved to {output_dir}/")
    
    return {
        'llm_analysis': llm_analysis,
        'human_analysis': human_analysis
    }

In [ ]:
results = run_complete_agreement_analysis(
    'expert_feedback_combined_llms.csv',
    hash_column='reaction_hash',
    output_dir='agreement_pie_charts'
)

In [ ]:
results = run_complete_agreement_analysis(
    'expert_feedback_combined_llms.csv', 
    hash_column='root_molecule_hash',
    category_column='general_feedback',
    expert_column='source_file',
    text_column='general_feedback_text',
    output_dir='agreement_pie_charts_routes',
)